# Unified Run — All 9 Cells

Owner: Mayukh (Transformer). COMP6242 'RNN's Revenge' paradigm comparison.

Protocol: **dropout 0.05, lr 3e-4, seed 42** across all tasks.
Dropout revised from v4.0 spec's 0.2 after ablation showed 0.2 suppresses long-range induction-head formation.

| Task | Lengths | Steps | Trainer |
|---|---|---|---|
| Shakespeare | 256 / 1024 / 2048 | 5K | `train.py` |
| Copy | short / medium / long | 5K | `train_synthetic.py` |
| Induction | short / medium / long | 20K (lr_decay 50K) | `train_synthetic.py` |

All runs write `out/<run_name>/summary.json`.

## Step 0: Sanity check

In [ ]:
import torch, subprocess
print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'bf16: {torch.cuda.is_bf16_supported()}')

# Confirm dataset_utils has the padding-mask fix (expect 2 lines)
n = subprocess.run(['grep', '-c', 'labels\\[seq_len', 'dataset_utils.py'],
                   capture_output=True, text=True).stdout.strip()
print(f'padding-mask lines in dataset_utils.py: {n}  (expect 2)')

## Step 1: Generate data (run once)

In [ ]:
# Generate induction data (corrected generator)
!python generate_induction.py
print('\nInduction data ready.')

In [ ]:
# Generate copy data
!python generate_longrange_copy.py
print('\nCopy data ready.')

In [ ]:
# Prepare Shakespeare data (download + tokenise, one-time)
!python data/tinyshakespeare/prepare.py
print('\nShakespeare data ready.')

## Step 2: Shakespeare × 3

In [ ]:
!python train.py config/tshake_256.py
!python train.py config/tshake_1024.py
!python train.py config/tshake_2048.py

## Step 3: Long-range Copy × 3

In [ ]:
!python train_synthetic.py config/copy_short.py
!python train_synthetic.py config/copy_medium.py
!python train_synthetic.py config/copy_long.py

## Step 4: Induction × 3 (20K steps, lr_decay 50K)

In [ ]:
!python train_synthetic.py config/induction_short.py

In [ ]:
!python train_synthetic.py config/induction_medium.py

In [ ]:
!python train_synthetic.py config/induction_long.py

## Step 5: Collect results

In [ ]:
import json, glob, math
from pathlib import Path

results = []
for p in sorted(glob.glob('out/*/summary.json')):
    with open(p) as f:
        s = json.load(f)
    row = {'run': s['run_name'], 'task': s.get('task', s.get('task_type', '?'))}
    if 'best_val_ppl' in s:
        row['val_ppl'] = round(s['best_val_ppl'], 4)
    if 'best_induction5_accuracy' in s:
        row['pat_acc5'] = round(s['best_induction5_accuracy'], 4)
        row['pattern_ppl'] = round(s.get('best_pattern_ppl', float('nan')), 4)
    if 'best_recall_ppl' in s:
        row['recall_ppl'] = round(s['best_recall_ppl'], 4)
    results.append(row)
    print(row)